# Course 6: Winning Online AI Competitions
## Session 9: Code competitions: inference under a time limit

**Goal:** build the parts of an offline inference notebook: a time budget, length-sorted batching, a fallback, and a dry run on fake data.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on synthetic or built-in data. Cells labelled NOT RUN need a Kaggle account, a competition's data, a GPU or an extra library.

### 1. What changes in a code competition
You submit a notebook, not a file. It runs on hidden test data with no internet, on fixed hardware, within a time limit (often 9 hours). If it crashes or times out near the end, you get no score and you have used a submission. So the engineering goal is: **always produce a valid submission in time**.

### 2. A stand-in model with realistic costs
Time per batch grows with the longest item in the batch, as it does for a transformer.

In [ ]:
import time, numpy as np, pandas as pd
rng = np.random.default_rng(0)
N = 2000
test = pd.DataFrame({"id": np.arange(N), "length": np.minimum(rng.lognormal(4.5, 0.9, N).astype(int) + 5, 2000)})     # token counts: many short, a few very long
SECONDS_PER_TOKEN = 2e-6                                           # pretend cost; a real model is thousands of times slower

def model_predict(batch_lengths, simulate=True):
    """Cost is (batch size) x (longest item): padding makes every row as long as the longest."""
    cost = len(batch_lengths) * max(batch_lengths) * SECONDS_PER_TOKEN
    if simulate: time.sleep(cost)
    return np.full(len(batch_lengths), 0.5), cost

### 3. Length-sorted batching
Batches of random rows are padded to their longest row. Sort by length first and each batch pads very little.

In [ ]:
def total_cost(order, batch_size=16):
    lengths = test.length.values[order]
    return sum(len(b) * b.max() for b in np.array_split(lengths, range(batch_size, len(lengths), batch_size))) * SECONDS_PER_TOKEN
random_order = rng.permutation(N); sorted_order = np.argsort(-test.length.values)                  # longest first: fail early if memory is short
c_rand, c_sort = total_cost(random_order), total_cost(sorted_order)
print(f"padded tokens processed: random order {c_rand / SECONDS_PER_TOKEN:,.0f} | sorted by length {c_sort / SECONDS_PER_TOKEN:,.0f} | saving {1 - c_sort / c_rand:.0%}")
assert c_sort < 0.7 * c_rand

Sort longest first. If a batch is going to run out of memory, it will be the first one, in the first minute, not the last one in hour eight. Remember to put predictions back in the original order.

### 4. A time budget with a fallback
Measure as you go. If the projected finish is past the deadline, switch to a cheaper path for the remaining rows (a smaller model, shorter truncation, or a constant) and still write a complete file.

In [ ]:
class Budget:
    def __init__(self, seconds, reserve=0.1): self.t0, self.limit = time.perf_counter(), seconds * (1 - reserve)
    def elapsed(self): return time.perf_counter() - self.t0
    def remaining(self): return self.limit - self.elapsed()

def run_inference(time_limit, batch_size=16, max_len=None):
    budget = Budget(time_limit); order = np.argsort(-test.length.values); preds = np.full(N, np.nan); used_fallback = 0
    done_tokens, spent = 0, 0.0
    for start in range(0, N, batch_size):
        idx = order[start:start + batch_size]; lengths = test.length.values[idx]
        if max_len: lengths = np.minimum(lengths, max_len)
        est = len(lengths) * lengths.max() * (spent / done_tokens if done_tokens else SECONDS_PER_TOKEN * 1.5)        # estimated cost of this batch
        rest = sum(len(order[s:s + batch_size]) * test.length.values[order[s]] for s in range(start + batch_size, N, batch_size))
        projected = est + rest * (spent / done_tokens if done_tokens else SECONDS_PER_TOKEN * 1.5)
        if projected > budget.remaining():
            preds[idx] = 0.5; used_fallback += len(idx); continue                           # cheap fallback for this batch; later batches are shorter and may fit
        p, cost = model_predict(lengths); preds[idx] = p; spent += cost; done_tokens += len(lengths) * lengths.max()
    sub = pd.DataFrame({"id": test.id, "prediction": preds})
    assert sub.prediction.notna().all() and len(sub) == N                                   # a complete, valid file, whatever happened
    return sub, budget.elapsed(), used_fallback

full_cost = total_cost(np.argsort(-test.length.values))
for name, limit in [("generous limit", full_cost * 3), ("tight limit", full_cost * 0.6)]:
    sub, t, fb = run_inference(limit)
    print(f"{name:15s} limit {limit:.2f}s | finished in {t:.2f}s | rows answered by the fallback: {fb} of {N}")
    assert t <= limit

Under the tight limit the longest rows fall back to the cheap answer and the rest are served by the model. A partial fallback costs a little score. A timeout costs all of it.

### 5. Truncation as a speed knob
Cutting very long inputs is often the best trade: a few rows lose some text, and total time drops a lot.

In [ ]:
for max_len in [None, 1024, 512, 256]:
    lengths = test.length.values if max_len is None else np.minimum(test.length.values, max_len)
    order = np.argsort(-lengths); cost = sum(len(b) * b.max() for b in np.array_split(lengths[order], range(16, N, 16))) * SECONDS_PER_TOKEN
    print(f"max length {str(max_len):>5s}: relative time {cost / full_cost:.2f} | rows truncated {0 if max_len is None else (test.length > max_len).mean():.1%}")

### 6. The dry run
The hidden test set is larger than the public sample, and sometimes has cases the sample lacks. Before submitting, run the whole notebook on a fake test set of the real size, with awkward rows added.

In [ ]:
def dry_run_cases(n):
    fake = pd.DataFrame({"id": np.arange(n), "text": ["word " * int(l) for l in np.minimum(rng.lognormal(4.5, 0.9, n).astype(int) + 5, 2000)]})
    fake.loc[0, "text"] = ""                                      # an empty input
    fake.loc[1, "text"] = "অ" * 5000                              # a very long input in another script
    fake.loc[2, "text"] = None                                    # a missing value
    return fake
def safe_length(t): return 1 if not isinstance(t, str) or not t.strip() else len(t.split())
fake = dry_run_cases(500); lengths = fake.text.map(safe_length)
print("rows:", len(fake), "| empty or missing inputs given a safe length:", int((lengths == 1).sum()), "| longest:", int(lengths.max()))
assert lengths.notna().all()

### 7. Checklist for the real notebook
1. Internet off. Every model, tokenizer and wheel comes from an attached dataset.
2. One cell at the top prints the time limit, the row count and the planned seconds per row.
3. Write a valid fallback submission **first**, then overwrite it when the real predictions are ready.
4. Inputs sorted by length; predictions restored to the original order by id.
5. `try / except` around each batch, with the fallback value on failure and a counter you print at the end.
6. A dry run on a full-size fake test set finishes within 80% of the limit.
7. Seeds fixed; library versions printed.

### 8. Faster LLM inference: 4-bit weights and vLLM (NOT RUN here)

In [ ]:
# NOT RUN: needs GPUs. In a Kaggle code competition, install from wheels you attached as a dataset (no internet).
# Option A: transformers + bitsandbytes 4-bit. Small memory use, simple, moderate speed.
#   model = AutoModelForCausalLM.from_pretrained(path, quantization_config=BitsAndBytesConfig(load_in_4bit=True), device_map="auto")
# Option B: vLLM. Continuous batching gives much higher throughput for generation; it accepts AWQ or GPTQ quantized weights.
#   from vllm import LLM, SamplingParams
#   llm = LLM(model=path, quantization="awq", dtype="half", gpu_memory_utilization=0.95, max_model_len=4096)
#   outs = llm.generate(prompts, SamplingParams(temperature=0.0, max_tokens=64))
# Measure both on 200 real-length prompts on the competition's hardware, in tokens per second, before choosing. Check the current docs for each argument.

### TODO
1. Make the notebook for your competition write a fallback submission in its first cell.
2. Add resume support: save predictions every 50 batches and skip finished ids on restart.
3. Measure real seconds per row for your model at three truncation lengths and fill in the budget table.